<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/Solar_Field_Transition_Analysis_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solar Field Transition Analysis v1.0.1
This notebook analyzes AIA 193 Å, 94 Å, and 304 Å images using entropy and ψ⋆s coherence across a DLSFH lattice to investigate potential dark photon transition signatures.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.filters import rank
from skimage.morphology import disk
from skimage.util import img_as_ubyte
from scipy.ndimage import gaussian_filter

In [3]:
from google.colab import files
uploaded = files.upload()
# Expected file names: aia193.png, aia94.png, aia304.png

Saving latest_1024_0304.jpg to latest_1024_0304.jpg
Saving latest_4096_0094.jpg to latest_4096_0094 (1).jpg
Saving latest_4096_0193.jpg to latest_4096_0193 (1).jpg


In [5]:
img_193 = np.array(Image.open("latest_4096_0193.jpg").convert('L'))
img_94 = np.array(Image.open("latest_4096_0094.jpg").convert('L'))
img_304 = np.array(Image.open("latest_1024_0304.jpg").convert('L'))

In [6]:
def compute_entropy(img):
    return rank.entropy(img_as_ubyte(img), disk(5)) / 255.0

entropy_193 = compute_entropy(img_193)
entropy_94 = compute_entropy(img_94)
entropy_304 = compute_entropy(img_304)

psi_193 = np.exp(-entropy_193)
psi_94 = np.exp(-entropy_94)
psi_304 = np.exp(-entropy_304)

In [12]:
# Get image dimensions
data = []
for i, (x, y) in enumerate(nodes):
    # Ensure (x, y) stay within image bounds
    x = np.clip(x, 0, width - 1)
    y = np.clip(y, 0, height - 1)

    s1 = entropy_193[y, x]
    s2 = entropy_94[y, x]
    s3 = entropy_304[y, x]

    p1, p2, p3 = np.exp(-s1), np.exp(-s2), np.exp(-s3)
    drop94 = p1 - p2
    drop304 = p1 - p3
    nonrec = drop94 > 0.05 and drop304 > 0.05

    data.append([i+1, x, y, s1, s2, s3, p1, p2, p3, drop94, drop304, nonrec])

df = pd.DataFrame(data, columns=[
    'Node', 'X', 'Y', 'S193', 'S94', 'S304', 'ψ193', 'ψ94', 'ψ304', 'Δψ94', 'Δψ304', 'NonRecondensing'
])
df



IndexError: index 2048 is out of bounds for axis 0 with size 1024

In [8]:
data = []
for i, (x, y) in enumerate(nodes):
    s1 = entropy_193[y,x]
    s2 = entropy_94[y,x]
    s3 = entropy_304[y,x]
    p1, p2, p3 = np.exp(-s1), np.exp(-s2), np.exp(-s3)
    drop94 = p1 - p2
    drop304 = p1 - p3
    nonrec = drop94 > 0.05 and drop304 > 0.05
    data.append([i+1,x,y,s1,s2,s3,p1,p2,p3,drop94,drop304,nonrec])
df = pd.DataFrame(data, columns=[
    'Node','X','Y','S193','S94','S304','ψ193','ψ94','ψ304','Δψ94','Δψ304','NonRecondensing'])
df

IndexError: index 2048 is out of bounds for axis 0 with size 1024

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(entropy_193, cmap='inferno')
for _, row in df.iterrows():
    color = 'red' if row['NonRecondensing'] else 'white'
    ax.plot(row['X'], row['Y'], 'o', color=color)
    ax.text(row['X']+4, row['Y'], str(row['Node']), color=color)
ax.set_title("Field Transition Zones (ψ⋆s Collapse + Non-Recondensing)")
ax.axis('off')
plt.show()